# **Cross-Dataset Experiment (DAWN → ACDC) — YOLOv11s**
- Training YOLOv11s on the DAWN dataset using COCO pretrained weights
- Evaluating the trained model on the global ACDC test set
- Performing weather-specific evaluations on the fog, rain, and snow subsets of the ACDC dataset

## Mount drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Ultralytics

In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00


## Import librairies

In [5]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import time
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Check GPU

In [6]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


## Path

In [7]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

RUNS_ROOT = PROJECT_ROOT / "Runs"
DATASETS_ROOT = PROJECT_ROOT / "Datasets" / "processed"

# DAWN (training dataset)
DAWN_YOLO_ROOT = DATASETS_ROOT / "dawn_yolo"

# ACDC (evaluation dataset)
ACDC_YOLO_ROOT = DATASETS_ROOT / "acdc_yolo"

## YAML Files

In [8]:
# DAWN training YAML
DAWN_DATA_YAML = DAWN_YOLO_ROOT / "dataset.yaml"

# ACDC evaluation YAMLs
ACDC_GLOBAL_YAML = ACDC_YOLO_ROOT / "acdc.yaml"
ACDC_FOG_YAML    = ACDC_YOLO_ROOT / "fog_only.yaml"
ACDC_RAIN_YAML   = ACDC_YOLO_ROOT / "rain_only.yaml"
ACDC_SNOW_YAML   = ACDC_YOLO_ROOT / "snow_only.yaml"

## Classes

In [9]:
CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

## Experiment Settings

In [10]:
SEED = 456

MODEL_NAME = "yolo11s.pt"

EXPERIMENT_NAME = f"dawn_to_acdc_yolo11s_seed{SEED}"

## Training

In [11]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DAWN_DATA_YAML),

    # Training
    epochs=100,
    patience=10,

    # Image / batch
    imgsz=640,
    batch=16,

    # Reproducibility
    seed=SEED,
    deterministic=True,
    pretrained=True,

    # Validation
    val=True,

    # Workers
    workers=2,

    # Output
    project=str(RUNS_ROOT / "yolo"),
    name=EXPERIMENT_NAME,
    exist_ok=True
)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dawn_to_acdc_yolo11s_see

## Load best model

In [12]:
BEST_MODEL_PATH = (
    RUNS_ROOT
    / "yolo"
    / EXPERIMENT_NAME
    / "weights"
    / "best.pt"
)

best_model = YOLO(str(BEST_MODEL_PATH))

print(f"\nBest model loaded from:\n{BEST_MODEL_PATH}")


Best model loaded from:
/content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/weights/best.pt


## Evaluation Function

In [13]:
def evaluate_split(model, yaml_path, split_name):

    metrics = model.val(
        data=str(yaml_path),
        split="test",
        verbose=False
    )

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)

    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    preprocess_ms = float(metrics.speed["preprocess"])
    inference_ms = float(metrics.speed["inference"])
    postprocess_ms = float(metrics.speed["postprocess"])

    total_ms_per_image = (
        preprocess_ms
        + inference_ms
        + postprocess_ms
    )

    fps = (
        1000 / total_ms_per_image
        if total_ms_per_image > 0
        else 0.0
    )

    return {
        "source_train_dataset": "DAWN",
        "target_test_dataset": "ACDC",
        "model": "YOLOv11s",
        "experiment": "DAWN->ACDC",
        "seed": SEED,
        "condition": split_name,

        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),

        "Precision": precision,
        "Recall": recall,
        "F1-score": f1_score,

        "Preprocess_ms_per_image": preprocess_ms,
        "Inference_ms_per_image": inference_ms,
        "Postprocess_ms_per_image": postprocess_ms,
        "Total_ms_per_image": total_ms_per_image,
        "FPS": fps,
    }

## Global and weather-specific evaluation

In [14]:
evaluation_configs = [
    ("global", ACDC_GLOBAL_YAML),
    ("fog", ACDC_FOG_YAML),
    ("rain", ACDC_RAIN_YAML),
    ("snow", ACDC_SNOW_YAML),
]

all_results = []

for split_name, yaml_path in evaluation_configs:

    print(f"/n Evaluating DAWN -> ACDC on: {split_name}")

    result = evaluate_split(
        model=best_model,
        yaml_path=yaml_path,
        split_name=split_name
    )

    all_results.append(result)

/n Evaluating DAWN -> ACDC on: global
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.1±0.4 ms, read: 0.4±0.2 MB/s, size: 477.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 174.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 2.6s/it 1:30
                   all        540       3358      0.519      0.208      0.193      0.107
Speed: 0.5ms preprocess, 1.8ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /

## Results dataframe

In [15]:
results_df = pd.DataFrame(all_results)

print("DAWN -> ACDC YOLOv11s RESULTS")

display(results_df)

DAWN -> ACDC YOLOv11s RESULTS


,source_train_dataset,target_test_dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,Total_ms_per_image,FPS
0,DAWN,ACDC,YOLOv11s,DAWN->ACDC,456,global,0.106570,0.193018,0.099632,0.518796,0.208402,0.297355,0.474133,1.822710,1.270102,3.566945,280.351925
1,DAWN,ACDC,YOLOv11s,DAWN->ACDC,456,fog,0.121725,0.220863,0.107612,0.465618,0.276483,0.346948,1.029645,6.420143,5.059261,12.509049,79.942129
2,DAWN,ACDC,YOLOv11s,DAWN->ACDC,456,rain,0.098351,0.175580,0.097122,0.508209,0.201338,0.288414,0.569640,1.403336,1.887669,3.860645,259.024059
3,DAWN,ACDC,YOLOv11s,DAWN->ACDC,456,snow,0.134458,0.244226,0.123091,0.614810,0.236134,0.341215,0.736353,2.277194,3.915641,6.929189,144.317038


## Save results

In [16]:
RESULTS_DIR = (
    RUNS_ROOT
    / "yolo"
    / EXPERIMENT_NAME
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

csv_path = RESULTS_DIR / "dawn_to_acdc_yolo11s_results.csv"
json_path = RESULTS_DIR / "dawn_to_acdc_yolo11s_results.json"
xlsx_path = RESULTS_DIR / "dawn_to_acdc_yolo11s_results.xlsx"

results_df.to_csv(csv_path, index=False)

results_df.to_json(
    json_path,
    orient="records",
    indent=4
)

results_df.to_excel(
    xlsx_path,
    index=False
)

print("\nResults saved:")
print(f"CSV  : {csv_path}")
print(f"JSON : {json_path}")
print(f"XLSX : {xlsx_path}")


Results saved:
CSV  : /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/dawn_to_acdc_yolo11s_results.csv
JSON : /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/dawn_to_acdc_yolo11s_results.json
XLSX : /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/dawn_to_acdc_yolo11s_results.xlsx


## Save summary

In [17]:
summary_path = RESULTS_DIR / "experiment_summary.txt"

with open(summary_path, "w") as f:
    f.write("YOLOv11s Cross-Dataset Experiment\n")
    f.write(f"Experiment: {EXPERIMENT_NAME}\n")
    f.write("Direction: DAWN -> ACDC\n")
    f.write("Source training dataset: DAWN\n")
    f.write("Target evaluation dataset: ACDC\n")
    f.write("Model: YOLOv11s\n")
    f.write(f"Seed: {SEED}\n")
    f.write("Training epochs: 100\n")
    f.write("Early stopping patience: 10\n")
    f.write("Image size: 640\n")
    f.write("Batch size: 16\n")
    f.write("Evaluation: global + fog + rain + snow\n")
    f.write(f"\nBest model path:\n{BEST_MODEL_PATH}\n")
    f.write(f"\nResults CSV:\n{csv_path}\n")
    f.write(f"\nResults JSON:\n{json_path}\n")
    f.write(f"\nResults XLSX:\n{xlsx_path}\n")

print(f"Summary saved: {summary_path}")

Summary saved: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/experiment_summary.txt
